# RAG with Ollama and Chroma

In [35]:
import ollama
from langchain_chroma import Chroma
from langchain_classic.chains.question_answering.map_reduce_prompt import messages
from langchain_core.documents import Document
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from sqlalchemy.sql.annotation import Annotated

In [37]:
document_loader = PyPDFDirectoryLoader('./data')
documents = document_loader.load()
print(f'Loaded {len(documents)} documents')

Loaded 1149 documents


In [38]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

last_page_id = None
current_chunk_index = 0
for chunk in chunks:
    source = chunk.metadata.get('source', 'unknown')
    page = chunk.metadata.get('page', 'number')
    current_page_id = f'{source}:{page}'
    if current_page_id == last_page_id:
        current_chunk_index += 1
    else:
        current_chunk_index = 0
    last_page_id = current_page_id
    chunk_id = f'{current_page_id}:{current_chunk_index}'
    chunk.metadata['id'] = chunk_id

#print(chunks[15])

page_content='as possible. This process assures the 
maximum transfer of both the words and 
the thoughts contained in the original.
The CSB uses optimal equivalence 
as its translation philosophy . In the 
many places throughout the Bible 
where a word-for-word rendering is 
understandable, a literal translation is 
used. When a word-for-word rendering 
might obscure the meaning for a modern 
audience, a more dynamic translation 
is used. The CSB places equal value on 
fidelity to the original and readability 
for a modern audience, resulting in a 
translation that achieves both goals.
The Gender Language Usage in Bible 
Translation
The goal of the translators of the CSB has 
not been to promote a cultural ideology but 
to translate the Bible faithfully . Recognizing 
modern usage of En glish, the CSB regular -
ly translates the plural of the Greek word 
ανθρωπος (“man”) as “people” instead of 
“men,” and occasionally the singular as “one,” 
“someone,” or “everyone,” when the supporti

In [39]:
embedding_model = OllamaEmbeddings(model='nomic-embed-text')
db = Chroma(collection_name='rag_datas', persist_directory='./data/database', embedding_function=embedding_model)

In [44]:
existing_items = db.get(include=[])
existing_ids = set(existing_items['ids'])
new_chunks = [chunk for chunk in chunks if chunk.metadata['id'] not in existing_ids]
new_ids = [chunk.metadata['id'] for chunk in new_chunks]
if len(new_ids) > 0:
    db.add_documents(documents=new_chunks[:3000], ids=new_ids[:3000])
    db.add_documents(documents=new_chunks[3000:], ids=new_ids[3000:])

In [50]:
bang_question = "How many player can play Bang?"
retriever = db.as_retriever(search_kwargs={'k': 3})
db_answer = retriever.invoke(question)
print(dbanswer[1].page_content)

2
PREPARATION
(Before the first game remove carefully the bullet tokens from their frames.)
Each player takes a playing board (place it in front of you to hold your role, 
your character, your weapon and your bullets).
Take as many role cards as the number of players, divided as follows:
 4 players: 1 Sheriff, 1 Renegade, 2 Outlaws
 5 players: 1 Sheriff, 1 Renegade, 2 Outlaws, 1 Deputy
 6 players: 1 Sheriff, 1 Renegade, 3 Outlaws, 1 Deputy
 7 players: 1 Sheriff, 1 Renegade, 3 Outlaws, 2 Deputy
Shuffle the cards and give one, face down, to each player. The Sheriff 
reveals himself by turning his card face up. All other players look at their 
role but keep it secret.
Shuffle the characters and give one face up to each player. 
Each player now announces the name of his character and 
reads his ability. Each player takes as many bullets as 
shown on his character.
The Sheriff plays the game with one additional bullet: if 
his character card shows three bullets, he is considered


# State Graph for RAG Chatbot

In [60]:
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated
from langchain_ollama.llms import OllamaLLM

In [61]:
llm = OllamaLLM(model='minimax-m2.5:cloud')

class ChatbotState(TypedDict):
    messages: Annotated[list, add_messages]

In [65]:
def chatbot_node(state: ChatbotState) ->  ChatbotState:
    return {'messages': [{'role': 'ai', 'content': llm.invoke(state['messages'])}]}

In [ ]:
def rag_node(state: ChatbotState) -> ChatbotState:
    retriever = db.as_retriever(search_kwargs={'k': 3})
    state['retrieved_chunks'] = retriever.invoke(state['question'])
    return state

In [ ]:
def web_search_node(state: ChatbotState) -> ChatbotState:
    # Placeholder for web search logic
    state['web_search_flag'] = True
    return state

In [66]:
workflow = StateGraph(ChatbotState)

workflow.add_node('chatbot', chatbot_node)

workflow.add_edge(START, 'chatbot')
workflow.add_edge('chatbot', END)

graph = workflow.compile()

In [72]:
state = None
while True:
    in_message = input("User: ")
    if in_message.lower().strip() in ['exit', 'quit']:
        break

    print(f"User: {in_message}")
    if state is None:
        state: ChatbotState = {'messages': [{'role': 'user', 'content': in_message}]}
    else:
        state['messages'].append({'role': 'user', 'content': in_message})

    state = graph.invoke(state)
    response = state['messages'][-1].content
    print(f"Bot: {response}")

User: who was the first person on the moon? one word
Bot: Armstrong
User: in which yeeaer?
Bot: 1969
User: thanks
Bot: You're welcome! �月球
User: im Levi
Bot: Nice to meet you, Levi! 👋

Is there anything I can help you with today?
User: bye
Bot: Goodbye, Levi! Take care! 👋
